# Notebook 04: Inference on evaluation targets

Loads each saved model checkpoint from Notebook 03 and runs inference on its corresponding cross-population target:
- K2N models (trained on Kermany) are evaluated on the Togunwa holdout set.
- N2K models (trained on Togunwa) are evaluated on the Kermany test set.

Saves raw logits and predicted probabilities to disk. Downstream analysis and calibration notebooks read these saved predictions so forward passes only run once.

## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import sys
from pathlib import Path

REPO_ON_DRIVE = "/content/drive/MyDrive/crosspop-cxr-asymmetry"
for candidate in [REPO_ON_DRIVE, "..", "."]:
    p = Path(candidate).resolve()
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break

import importlib
import pandas as pd
import numpy as np
import torch

import config
from src import splits as SP
from src import datasets as DS
from src import models as M
from src import inference as INF
from src import metrics as MET
from src import results as RES

for mod in (config, SP, DS, M, INF, MET, RES):
    importlib.reload(mod)

config.ensure_output_dirs()

device = config.get_device()


Mounted at /content/drive
Compute Device: GPU (Tesla T4)


In [ ]:
PRED_DIR = config.RESULTS_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = config.RESULTS_DIR / "manifests"
manifest = pd.read_csv(MANIFEST_DIR / "splits_all.csv")
eval_targets = pd.read_csv(MANIFEST_DIR / "eval_targets.csv")

print(f"Manifest shape: {manifest.shape} | Eval targets shape: {eval_targets.shape}")


Manifest shape: (4134, 10) | Eval targets shape: (624, 10)


## 2. Evaluation Target Datasets

In [ ]:
tf_eval = DS.build_transforms(
    config.IMAGE_SIZE,
    config.IMAGENET_MEAN,
    config.IMAGENET_STD,
    train=False
)

def build_togunwa_holdout_ds(seed):
    """Build dataset for Togunwa holdout evaluation given a seed."""
    rows = manifest[
        (manifest.dataset == "togunwa") &
        (manifest.split == "holdout") &
        (manifest.seed.astype(str) == str(seed))
    ]
    ds = DS.CXRDataset(rows[["path", "label"]].reset_index(drop=True), tf_eval)
    return ds, len(rows)

def build_kermany_test_ds():
    """Build dataset for the global Kermany test set evaluation."""
    rows = eval_targets[
        (eval_targets.dataset == "kermany") &
        (eval_targets.split == "test")
    ]
    ds = DS.CXRDataset(rows[["path", "label"]].reset_index(drop=True), tf_eval)
    return ds, len(rows)

kermany_test_dataset, n_kermany = build_kermany_test_ds()
print(f"\nKermany test set size: {n_kermany} (N2K target)")

for seed in config.SEEDS:
    _, n_togunwa = build_togunwa_holdout_ds(seed)
    print(f"Togunwa holdout seed {seed} size: {n_togunwa} (K2N target)")



Kermany test set size: 624 (N2K target)
Togunwa holdout seed 42 size: 30 (K2N target)
Togunwa holdout seed 43 size: 30 (K2N target)
Togunwa holdout seed 44 size: 30 (K2N target)


## 3. Execute Inference across Checkpoints

In [ ]:
checkpoints = sorted(config.CHECKPOINTS_DIR.glob("*.pth"))
print(f"\nFound {len(checkpoints)} checkpoint file(s) for evaluation.")

records = []

for ckpt in checkpoints:
    info = RES.parse_run_id(ckpt.stem)
    arch = info["arch"]
    direction = info["direction"]
    seed = info["seed"]

    model = M.build_model(arch, pretrained=False)
    model.load_state_dict(torch.load(ckpt, map_location=device))

    if direction == "K2N":
        eval_ds, _ = build_togunwa_holdout_ds(seed)
        target = "togunwa_holdout"
    else:
        eval_ds = kermany_test_dataset
        target = "kermany_test"

    out = INF.predict(
        model,
        eval_ds,
        device,
        batch_size=32,
        temperature=1.0,
        num_workers=2
    )

    # Export raw prediction probabilities and logits for downstream analysis
    pred_df = pd.DataFrame({
        "y_true": out["y_true"],
        "y_prob": out["y_prob"],
        "logit_0": out["logits"][:, 0],
        "logit_1": out["logits"][:, 1]
    })

    out_csv_path = PRED_DIR / f"{ckpt.stem}__on__{target}.csv"
    pred_df.to_csv(out_csv_path, index=False)

    # Calculate performance metrics using project evaluation utilities
    m = MET.all_metrics(
        out["y_true"],
        out["y_prob"],
        fixed_sensitivity=config.FIXED_SENSITIVITY,
        n_bins=config.ECE_N_BINS
    )

    records.append({
        **info,
        "target": target,
        "n": m["n"],
        "n_pos": m["n_positive"],
        "auroc": m["auroc"],
        "spec@95": m["specificity_at_95_sens"],
        "ece": m["ece"],
        "ece_adaptive": m["ece_adaptive"],
        "brier": m["brier"]
    })

    print(f"{ckpt.stem[:48]:48s} AUROC={m['auroc']:.3f} ECE_Std={m['ece']:.3f} ECE_Adp={m['ece_adaptive']:.3f}")

print(f"\nSaved predictions for {len(records)} runs to {PRED_DIR.name}/")



Found 48 checkpoint file(s) for evaluation.
efficientnet_b0__K2N__size100__seed42            AUROC=0.498 ECE_Std=0.365 ECE_Adp=0.389
efficientnet_b0__K2N__size100__seed43            AUROC=0.587 ECE_Std=0.382 ECE_Adp=0.352
efficientnet_b0__K2N__size100__seed44            AUROC=0.769 ECE_Std=0.250 ECE_Adp=0.243
efficientnet_b0__K2N__size190__seed42            AUROC=0.489 ECE_Std=0.383 ECE_Adp=0.445
efficientnet_b0__K2N__size190__seed43            AUROC=0.604 ECE_Std=0.358 ECE_Adp=0.394
efficientnet_b0__K2N__size190__seed44            AUROC=0.787 ECE_Std=0.247 ECE_Adp=0.270
efficientnet_b0__K2N__size50__seed42             AUROC=0.533 ECE_Std=0.354 ECE_Adp=0.365
efficientnet_b0__K2N__size50__seed43             AUROC=0.507 ECE_Std=0.314 ECE_Adp=0.330
efficientnet_b0__K2N__size50__seed44             AUROC=0.720 ECE_Std=0.220 ECE_Adp=0.299
efficientnet_b0__N2K__size190__seed42__fold0     AUROC=0.650 ECE_Std=0.206 ECE_Adp=0.204
efficientnet_b0__N2K__size190__seed42__fold1     AUROC=0.845 ECE_

## 4. Save and Display Summary Results

In [ ]:
summary = pd.DataFrame(records)

if len(summary) > 0:
    base_cols = ["arch", "direction", "size", "seed", "fold", "target", "n", "n_pos", "auroc", "spec@95", "ece", "ece_adaptive", "brier"]
    cols = [c for c in base_cols if c in summary.columns]

    # Sort deterministically for clean tabular output
    sort_cols = [c for c in ["arch", "direction", "size", "seed", "fold"] if c in summary.columns]
    summary = summary[cols].sort_values(sort_cols)

    summary_path = config.RESULTS_DIR / "04_inference_summary.csv"
    summary.to_csv(summary_path, index=False)

    print("\n--- Inference Summary Table ---")
    print(summary.to_string(index=False))

    print("\n--- Mean Metrics by Transfer Direction ---")
    print(summary.groupby("direction")[["auroc", "ece", "ece_adaptive"]].agg(["mean", "count"]).round(3).to_string())
else:
    print("\nNo checkpoints found. Please run Notebook 03 before executing inference.")



--- Inference Summary Table ---
           arch direction  size  seed  fold          target   n  n_pos    auroc  spec@95      ece  ece_adaptive    brier
efficientnet_b0       K2N    50    42   NaN togunwa_holdout  30     15 0.533333 0.000000 0.354396      0.364977 0.336351
efficientnet_b0       K2N    50    43   NaN togunwa_holdout  30     15 0.506667 0.200000 0.313690      0.329766 0.325418
efficientnet_b0       K2N    50    44   NaN togunwa_holdout  30     15 0.720000 0.066667 0.220441      0.299054 0.228584
efficientnet_b0       K2N   100    42   NaN togunwa_holdout  30     15 0.497778 0.000000 0.364981      0.389316 0.337968
efficientnet_b0       K2N   100    43   NaN togunwa_holdout  30     15 0.586667 0.200000 0.381695      0.351683 0.322027
efficientnet_b0       K2N   100    44   NaN togunwa_holdout  30     15 0.768889 0.333333 0.250058      0.243423 0.228231
efficientnet_b0       K2N   190    42   NaN togunwa_holdout  30     15 0.488889 0.000000 0.382534      0.444705 0.352880

**Next:** notebook 05 (discrimination, SQ1).